In [ ]:
from sentence_transformers import SentenceTransformer

# Load a small, fast embedding model we will use for semantic similarity.
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Example question and its embedding vector.
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1

In [ ]:
# A candidate answer text and its embedding vector.
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
dv

In [ ]:
# Dot product gives a quick similarity score between question and answer.
v1.dot(dv)

In [ ]:
# A different question, likely less related to the same answer.
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [ ]:
# Compare the second question against the same answer vector.
v2.dot(dv)

In [ ]:
from ingest import load_faq_data

# Load FAQ documents (question/answer pairs) from the ingest pipeline.
documents = load_faq_data()

In [ ]:
# Join question and answer into one text per document for embedding.
texts = []

for doc in documents:
    text = doc["question"] + "    " + doc["answer"]
    texts.append(text)

In [ ]:
# Quick preview to verify the combined text looks right.
texts[:5]

In [ ]:
# Progress bar helper for batch processing.
from tqdm.auto import tqdm

In [ ]:
# Encode texts in batches to avoid big memory spikes.
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i: i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

In [ ]:
import numpy as np

# Convert list of vectors to a 2D NumPy array for fast math ops.
X = np.array(vectors)

In [ ]:
# Shape should be: (number_of_docs, embedding_dim).
X.shape

In [ ]:
# Encode the user query we want to search with.
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [ ]:
# Vectorized similarity: score every document against the query.
scores = X.dot(v_query)

In [ ]:
# Same scoring idea written explicitly with a loop.
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [ ]:
# Index of the single best match and its score.
idx = np.argmax(scores)
idx, scores[idx]

In [ ]:
# Inspect the top document content.
documents[idx]

In [ ]:
# Get indices of the 5 highest scores (unsorted ascending slice).
top5 = np.argsort(scores)[-5:]

In [ ]:
# Reverse to show best-to-worst among top 5.
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

# Same result in one line.
topk = np.argsort(scores)[-5:][::-1]

In [ ]:
# Print each top match with its score for manual inspection.
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

In [24]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [25]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [30]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [27]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [28]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}